In [1]:

from jupyter_lab_notebook_toc_utils import generate_toc, display_toc
toc = generate_toc(add_numbering=True)
display_toc(toc)

**Table of Contents**<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**1.** **Overview**](#Overview)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**2.** **History**](#History)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**3.** **SuperAgent**](#SuperAgent)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.** Installation](#Installation)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.1.** Clone The Repo](#Clone-The-Repo)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.2.** Edit the hosts file](#Edit-the-hosts-file)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.3.** Deploy Supabase](#Deploy-Supabase)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.4.** Deploy LangFuse](#Deploy-LangFuse)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**4.** **Langfuse tracing**](#Langfuse-tracing)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.1.** Deploy Vector Store](#Deploy-Vector-Store)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**5.** **Weaviate vars**](#Weaviate-vars)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**5.1.** Deploy Superagent](#Deploy-Superagent)

# Overview

While I was able to get SuperAgent to install, I was unable to get SuperAgent to work properly. I have opened a [bug](https://github.com/superagent-ai/superagent/issues/959) on github.

# History

A user named homanp originally wrote a solution called [LangChain UI](https://github.com/homanp/langchain-ui). In May 2023, he [announced on X](https://twitter.com/pelaseyed/status/1658112221937909763) that he has rewritten the backend and rebranded / released a new solution called superagent. The LangChain UI notes that the repo is depricated in facor of superagent.

# SuperAgent

## Installation

For this installation I will be self hosting on a machine running Docker. The [official documentation](https://docs.superagent.sh/overview/installation/running-locally) lists the instructions for getting this solution stood up, but from what I have seen, they are missing a few steps. I will elaborate as we go.

The website lists the following components as being included in the deployment:

1. Supabase - Database required for the Superagent UI, RAG file storage and UI Authentication.
2. Superagent API - The API for Superagent along with its own postgres db for storage.
3. Superagent UI - A user interface for interacting with Superagent.
4. Motorhead - A redis based memory store for LLM interactions
5. Langfuse - Open source Observabillity & Analytics for LLM Apps
6. A vector store - Choose from a selection of supported vector stores for performing Retrieval Augmented Generation.

The installation is informed through a collection of .env files which define BASH environment variables that get consumed when we run the automation. I will touch on this more in the future.

### Clone The Repo
The first setp is to clone the repo and checkout the particular version of code we want to run. Looking at their release log, v0.2.28 is the latest version. I will make sure to checkout the repository to that version using the tag provided by the repo administrators.

```
(base) [root@fedora ~]# git clone git@github.com:homanp/superagent.git
(base) [root@fedora ~]# cd superagent
(base) [root@fedora superagent]# git checkout v0.2.28
```

### Edit the hosts file

We need to configure the VM so that it knows the IP addresses for various services that appear to be hardcoded in the solution. We must add the following lines to our /etc/hosts file:

```
127.0.0.1  superagent-api
127.0.0.1  superagent-ui
127.0.0.1  pgdb
127.0.0.1  pgadmin
127.0.0.1  supabase-kong
127.0.0.1  langfuse-server
127.0.0.1  redis
127.0.0.1  weaviate
127.0.0.1  qdrant
```

### Deploy Supabase

Supabase is [billed](https://supabase.com/) as an opensource FireBase alternative. It is essentially a framework offering backend cloud computing services and application development platforms. According to the [superagent documentation about supabase](https://docs.superagent.sh/overview/dependencies/supabase), supabase provides the database, Authentication with well know auth providers and a Bucket for file storage.

The first thing to do is create the config file with the environment variables. If we have a look at this file we will see a number of configuration parameters are available. In my case I accepted the defaults and disabled the the external github authentication. In this situation I did not want to tie my system to any external providers. 

**Note**: You should definately change the default secrets for a more secure deployment.

```
(base) [root@fedora superagent]# cd libs/.docker/ui/supabase/
(base) [root@fedora supabase]# cp .env.example .env
(base) [root@fedora supabase]# diff .env .env.example
77c77
< GOTRUE_EXTERNAL_GITHUB_ENABLED=false
---
> GOTRUE_EXTERNAL_GITHUB_ENABLED=true
```

We can then start supabase using the provided run.sh script which in turn relies on docker compose.

```
(base) [root@fedora supabase]# chmod +x *.sh
(base) [root@fedora supabase]# ./run.sh
```

This command will output a large amount of text informing what the script is doing. In a nutshell, docker compose will spin up a number of containers which provide various backend service.

```
[+] Running 12/12
 ✔ Container supabase-imgproxy               Started                                                             1.8s
 ✔ Container supabase-vector                 Started                                                             1.7s
 ✔ Container supabase-db                     Created                                                             0.9s
 ✔ Container supabase-analytics              Created                                                             0.5s
 ✔ Container realtime-dev.supabase-realtime  Created                                                             1.7s
 ✔ Container supabase-edge-functions         Created                                                             1.7s
 ✔ Container supabase-auth                   Created                                                             1.7s
 ✔ Container supabase-studio                 Created                                                             1.4s
 ✔ Container supabase-meta                   Created                                                             1.7s
 ✔ Container supabase-rest                   Created                                                             1.4s
 ✔ Container supabase-kong                   Created                                                             1.4s
 ✔ Container supabase-storage                Created                                                             1.1s
```

We then confirm we can access the supabase UI on internal and external IPs. The UI will prompt you for the username and password configured through the `libs/.docker/ui/supabase/.env` file. In the default case the credentials are: supabase/this_password_is_insecure_and_should_be_updated.

<center><img src="images/supabase_ui_first_login.png"></center>

According to the [documentation](https://docs.superagent.sh/overview/dependencies/supabase#supabase-storage), Next we need to perform a manual step to enable supabase to access the file storage that was previously configured. In my cases the bucket was public by default so maybe the documentation needs updating?

<center><img src="images/supabase_storage.png" style="width:80%"></center>
<center><img src="images/supabase_storage_2.png" style="width:80%"></center>
<center><img src="images/supabase_storage_3.png" style="width:50%"></center>

### Deploy LangFuse

According to the [official site](https://langfuse.com/docs), LangFuse is an open source LLM engineering platform providing Traces, evals, prompt management and metrics for debuging, analizing and improving an based LLM application.

**Note** I will be exploring this technology specifically in a later article.

According to the [superagent documentation](https://docs.superagent.sh/overview/logging/langfuse)

> Langfuse can be used with Superagent to enable detailed production traces and a granular view on quality, cost and latency.



The process for setting this up is very similar to that of supabase thanks to the prebuilt automation.

```
(base) [root@fedora supabase]# cd /root/superagent/
(base) [root@fedora superagent]# cd libs/.docker/external/observabillity/langfuse/
(base) [root@fedora langfuse]# cp .env.example .env
(base) [root@fedora langfuse]# chmod +x *.sh
(base) [root@fedora langfuse]# ./run.sh
[+] Running 2/2
 ✔ Container langfuse-server  Removed                                                                            0.0s
 ✔ Container langfuse-db      Removed                                                                            1.8s
[+] Running 2/2
 ✔ Container langfuse-db      Healthy                                                                            0.7s
 ✔ Container langfuse-server  Started                                                                            0.7s
```

We can now check the web UI to see if the service is available.

<center><img src="images/langfuse_ui.png" style="width:100%"></center>

We then Create a project named superagent.

Then, Under settings and API Keys - create a new API Key, copy the Public Key and Secret Key

<center><img src="images/langfuse_ui_api_key.png" style="width:100%"></center>
<center><img src="images/langfuse_ui_api_key_2.png" style="width:50%"></center>

We can then put these values into `libs/.docker/.env` file as follows:

```
# Langfuse tracing
LANGFUSE_PUBLIC_KEY=sk-lf-023b3c56-4609-afc7-42963e7181c2
LANGFUSE_SECRET_KEY=sk-lf-91a32ec6-8007-4449-a757-bc4911bff7cd
LANGFUSE_HOST=http://15.4.50.102:3100 ```

**Note** This file will be used later to deploy superagent (after all the base services are online).

### Deploy Vector Store
We need a database to store embedding vectors for RAG implementations and other Generative AI solution patterns. Superagent provides a guide for [Weaviate](https://docs.superagent.sh/overview/vector-stores/weaviate) with a promise that more guides will be coming soon. In the default configuration file we see additional mentions of pinecone and qdrant.

```
(base) [root@fedora superagent]# cd /root/superagent/
(base) [root@fedora superagent]# cd libs/.docker/external/vector-store/weaviate/
(base) [root@fedora weaviate]# cp .env.example .env
(base) [root@fedora weaviate]# chmod +x *.sh
(base) [root@fedora weaviate]# ./run.sh
[+] Running 1/1
 ✔ Container weaviate  Removed                                                                                   1.1s
[+] Running 1/1
 ✔ Container weaviate  Started                                                                                   0.9s
```

We can then check the web UI to see if the service is online

<center><img src="images/weaviate_ui.png" style="width:100%"></center>

We can now extract important information about the vector store and put that information in the superagent configuration file. We need the `AUTHENTICATION_APIKEY_ALLOWED_KEYS` key stored in the .env file and other variables to be updated in `libs/.docker/.env`.

```
(base) [root@fedora weaviate]# cat .env | grep AUTHENTICATION_APIKEY_ALLOWED_KEYS
      AUTHENTICATION_APIKEY_ALLOWED_KEYS="InsecurePassword"
(base) [root@fedora weaviate]# cd /root/superagent/
(base) [root@fedora superagent]# cat libs/.docker/.env
...
VECTORSTORE=weaviate # pinecone # `qdrant`, `weaviate` etc.
...
# Weaviate vars
WEAVIATE_API_KEY=InsecurePassword
WEAVIATE_INDEX=superagent
WEAVIATE_URL=http://15.4.50.102:8082

```

### Deploy Superagent

Now that the base level services are setup we can deploy superagenet. This process will also standup a punch of containers, some of which are additional other background services including:

* superagent-api - Superagent API
* superagent-ui - Superagent user interface
* motorhead- Motorhead redis based memory for llm interaction
* redis - redis backend for Motorhead
* pgdb - Postgres DB for the Superagent API
* pgadmin - User interface for the Postgres DB

We follow the same basic instructions to build and start containers.

**Note**: The build takes a while and timed out several times. I just kept rerunning the script until all the containers were started successfully.

```
(base) [root@fedora superagent]# cd /root/superagent/
(base) [root@fedora superagent]# cd libs/.docker
(base) [root@fedora .docker]# chmod +x *.sh
(base) [root@fedora .docker]# ./run.sh
```

We can then check that the services are running:

* Superagent UI: http://localhost:3000
* Superagent API: http://localhost:8080/api/v1
* Supabase Studio: http://localhost:8000
* Langfuse: http://localhost:3100